In [1]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("MILESTONE 2: HATE SPEECH DETECTION WITH FEATURE ENGINEERING")
print("Student: Aby Babu Alappat")
print("=" * 80)


# STEP 1: LOAD AND PREPARE DATA FROM MILESTONE 1

print("\n" + "=" * 80)
print("STEP 1: LOADING AND PREPARING DATA")
print("=" * 80)

# Load the dataset
df = pd.read_csv('hateXplain.csv')
print(f"Dataset loaded successfully!")
print(f"Total records: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# Basic preprocessing from milestone 1
df_clean = df.copy()
df_clean['target'].fillna('Unknown', inplace=True)
df_clean = df_clean.dropna(subset=['post_tokens'])
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.reset_index(drop=True)

print(f"\nAfter cleaning:")
print(f"Total records: {len(df_clean)}")

# Encode labels
le_label = LabelEncoder()
df_clean['label_encoded'] = le_label.fit_transform(df_clean['label'])

print(f"\nLabel distribution:")
print(df_clean['label'].value_counts())
print(f"\nLabel encoding:")
for i, label in enumerate(le_label.classes_):
    print(f"  {label}: {i}")


# STEP 2: TEXT FEATURE EXTRACTION USING TF-IDF

print("\n" + "=" * 80)
print("STEP 2: TEXT FEATURE EXTRACTION USING TF-IDF")
print("=" * 80)

# Create TF-IDF features from post_tokens
tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2),
    stop_words='english'
)

# Transform text to TF-IDF features
X_tfidf = tfidf_vectorizer.fit_transform(df_clean['post_tokens'])
X_tfidf_dense = X_tfidf.toarray()

print(f"TF-IDF Feature Matrix Shape: {X_tfidf_dense.shape}")
print(f"Number of features extracted: {X_tfidf_dense.shape[1]}")
print(f"Sample feature names: {list(tfidf_vectorizer.get_feature_names_out()[:10])}")

# Target variable
y = df_clean['label_encoded']


# STEP 3: BASELINE MODEL (NO FEATURE ENGINEERING)

print("\n" + "=" * 80)
print("STEP 3: BASELINE MODEL (NO FEATURE ENGINEERING)")
print("=" * 80)

# Split data
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_tfidf_dense, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train_base.shape[0]}")
print(f"Testing set size: {X_test_base.shape[0]}")

# Train Random Forest classifier
rf_clf_base = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
print(f"\nTraining Random Forest with {X_train_base.shape[1]} features...")
rf_clf_base.fit(X_train_base, y_train_base)

# Predictions
y_pred_base = rf_clf_base.predict(X_test_base)

# Evaluate
accuracy_base = accuracy_score(y_test_base, y_pred_base)
precision_base = precision_score(y_test_base, y_pred_base, average='weighted')
recall_base = recall_score(y_test_base, y_pred_base, average='weighted')
f1_base = f1_score(y_test_base, y_pred_base, average='weighted')

print(f"\nBaseline Results (All 1000 TF-IDF Features):")
print(f"  Accuracy:  {accuracy_base:.4f} ({accuracy_base*100:.2f}%)")
print(f"  Precision: {precision_base:.4f}")
print(f"  Recall:    {recall_base:.4f}")
print(f"  F1-Score:  {f1_base:.4f}")


# STEP 4: FEATURE ENGINEERING METHOD 1 - PCA

print("\n" + "=" * 80)
print("STEP 4: FEATURE ENGINEERING METHOD 1 - PCA")
print("=" * 80)

pca_components = [50, 100, 200]
pca_results = {}

for n_comp in pca_components:
    print(f"\n{'='*70}")
    print(f"Testing PCA with {n_comp} components")
    print(f"{'='*70}")

    # Initialize PCA
    pca = PCA(n_components=n_comp, random_state=42)

    # Transform features
    X_pca = pca.fit_transform(X_tfidf_dense)

    # Calculate explained variance
    explained_var = np.sum(pca.explained_variance_ratio_)

    print(f"Explained Variance: {explained_var:.4f} ({explained_var*100:.2f}%)")
    print(f"Reduced Feature Shape: {X_pca.shape}")

    # Split data
    X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
        X_pca, y, test_size=0.2, random_state=42, stratify=y
    )

    # Train Random Forest classifier
    rf_clf_pca = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_clf_pca.fit(X_train_pca, y_train_pca)

    # Predictions
    y_pred_pca = rf_clf_pca.predict(X_test_pca)

    # Evaluate
    accuracy_pca = accuracy_score(y_test_pca, y_pred_pca)
    precision_pca = precision_score(y_test_pca, y_pred_pca, average='weighted')
    recall_pca = recall_score(y_test_pca, y_pred_pca, average='weighted')
    f1_pca = f1_score(y_test_pca, y_pred_pca, average='weighted')

    print(f"\nModel Performance:")
    print(f"  Accuracy:  {accuracy_pca:.4f} ({accuracy_pca*100:.2f}%)")
    print(f"  Precision: {precision_pca:.4f}")
    print(f"  Recall:    {recall_pca:.4f}")
    print(f"  F1-Score:  {f1_pca:.4f}")
    print(f"  Difference from baseline: {(accuracy_pca - accuracy_base)*100:+.2f}%")

    # Store results
    pca_results[n_comp] = {
        'explained_variance': explained_var,
        'accuracy': accuracy_pca,
        'precision': precision_pca,
        'recall': recall_pca,
        'f1_score': f1_pca,
        'X_transformed': X_pca,
        'model': rf_clf_pca,
        'pca_object': pca
    }

# Select best PCA configuration
best_pca_n = max(pca_results.keys(), key=lambda k: pca_results[k]['accuracy'])
print(f"\n{'='*70}")
print(f"✓ Best PCA Configuration: {best_pca_n} components")
print(f"  Accuracy: {pca_results[best_pca_n]['accuracy']:.4f}")
print(f"  Explained Variance: {pca_results[best_pca_n]['explained_variance']:.4f}")
print(f"{'='*70}")


# STEP 5: FEATURE ENGINEERING METHOD 2 - LDA

print("\n" + "=" * 80)
print("STEP 5: FEATURE ENGINEERING METHOD 2 - LDA")
print("=" * 80)

# LDA can only extract up to (n_classes - 1) components
n_classes = len(np.unique(y))
max_lda_components = n_classes - 1

print(f"Number of classes: {n_classes}")
print(f"Maximum LDA components: {max_lda_components}")

print(f"\n{'='*70}")
print(f"Applying LDA with {max_lda_components} components")
print(f"{'='*70}")

# Apply LDA
lda = LDA(n_components=max_lda_components)
X_lda = lda.fit_transform(X_tfidf_dense, y)

print(f"LDA Feature Shape: {X_lda.shape}")
print(f"Explained Variance Ratio: {lda.explained_variance_ratio_}")
print(f"Total Explained Variance: {np.sum(lda.explained_variance_ratio_):.4f}")

# Split data
X_train_lda, X_test_lda, y_train_lda, y_test_lda = train_test_split(
    X_lda, y, test_size=0.2, random_state=42, stratify=y
)

# Train Random Forest classifier
rf_clf_lda = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf_lda.fit(X_train_lda, y_train_lda)

# Predictions
y_pred_lda = rf_clf_lda.predict(X_test_lda)

# Evaluate
accuracy_lda = accuracy_score(y_test_lda, y_pred_lda)
precision_lda = precision_score(y_test_lda, y_pred_lda, average='weighted')
recall_lda = recall_score(y_test_lda, y_pred_lda, average='weighted')
f1_lda = f1_score(y_test_lda, y_pred_lda, average='weighted')

print(f"\nModel Performance:")
print(f"  Accuracy:  {accuracy_lda:.4f} ({accuracy_lda*100:.2f}%)")
print(f"  Precision: {precision_lda:.4f}")
print(f"  Recall:    {recall_lda:.4f}")
print(f"  F1-Score:  {f1_lda:.4f}")
print(f"  Difference from baseline: {(accuracy_lda - accuracy_base)*100:+.2f}%")

lda_results = {
    'n_components': max_lda_components,
    'explained_variance': np.sum(lda.explained_variance_ratio_),
    'accuracy': accuracy_lda,
    'precision': precision_lda,
    'recall': recall_lda,
    'f1_score': f1_lda,
    'X_transformed': X_lda,
    'model': rf_clf_lda,
    'lda_object': lda
}


# STEP 6: FEATURE ENGINEERING METHOD 3 - CHI-SQUARE TEST

print("\n" + "=" * 80)
print("STEP 6: FEATURE ENGINEERING METHOD 3 - CHI-SQUARE TEST")
print("=" * 80)

chi2_k_values = [50, 100, 200, 500]
chi2_results = {}

for k in chi2_k_values:
    if k > X_tfidf_dense.shape[1]:
        k = X_tfidf_dense.shape[1]

    print(f"\n{'='*70}")
    print(f"Testing Chi-Square with k={k} best features")
    print(f"{'='*70}")

    # Initialize Chi-Square selector
    chi2_selector = SelectKBest(chi2, k=k)

    # Transform features
    X_chi2 = chi2_selector.fit_transform(X_tfidf_dense, y)

    print(f"Selected Feature Shape: {X_chi2.shape}")

    # Get selected feature indices and names
    selected_indices = chi2_selector.get_support(indices=True)
    feature_names = tfidf_vectorizer.get_feature_names_out()
    selected_features = feature_names[selected_indices]

    print(f"Top 10 selected features: {list(selected_features[:10])}")

    # Split data
    X_train_chi2, X_test_chi2, y_train_chi2, y_test_chi2 = train_test_split(
        X_chi2, y, test_size=0.2, random_state=42, stratify=y
    )

    # Train Random Forest classifier
    rf_clf_chi2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_clf_chi2.fit(X_train_chi2, y_train_chi2)

    # Predictions
    y_pred_chi2 = rf_clf_chi2.predict(X_test_chi2)

    # Evaluate
    accuracy_chi2 = accuracy_score(y_test_chi2, y_pred_chi2)
    precision_chi2 = precision_score(y_test_chi2, y_pred_chi2, average='weighted')
    recall_chi2 = recall_score(y_test_chi2, y_pred_chi2, average='weighted')
    f1_chi2 = f1_score(y_test_chi2, y_pred_chi2, average='weighted')

    print(f"\nModel Performance:")
    print(f"  Accuracy:  {accuracy_chi2:.4f} ({accuracy_chi2*100:.2f}%)")
    print(f"  Precision: {precision_chi2:.4f}")
    print(f"  Recall:    {recall_chi2:.4f}")
    print(f"  F1-Score:  {f1_chi2:.4f}")
    print(f"  Difference from baseline: {(accuracy_chi2 - accuracy_base)*100:+.2f}%")

    # Store results
    chi2_results[k] = {
        'k_features': k,
        'accuracy': accuracy_chi2,
        'precision': precision_chi2,
        'recall': recall_chi2,
        'f1_score': f1_chi2,
        'X_transformed': X_chi2,
        'model': rf_clf_chi2,
        'selector': chi2_selector,
        'selected_features': selected_features
    }

# Select best Chi-Square configuration
best_chi2_k = max(chi2_results.keys(), key=lambda k: chi2_results[k]['accuracy'])
print(f"\n{'='*70}")
print(f"✓ Best Chi-Square Configuration: k={best_chi2_k} features")
print(f"  Accuracy: {chi2_results[best_chi2_k]['accuracy']:.4f}")
print(f"{'='*70}")


# STEP 7: COMPREHENSIVE COMPARISON OF ALL METHODS

print("\n" + "=" * 80)
print("STEP 7: COMPREHENSIVE COMPARISON OF ALL METHODS")
print("=" * 80)

# Create comparison dataframe
comparison_data = {
    'Method': [
        'Baseline (Original TF-IDF)',
        f'PCA ({best_pca_n} components)',
        f'LDA ({max_lda_components} components)',
        f'Chi-Square (k={best_chi2_k})'
    ],
    'Features': [
        X_tfidf_dense.shape[1],
        best_pca_n,
        max_lda_components,
        best_chi2_k
    ],
    'Accuracy': [
        accuracy_base,
        pca_results[best_pca_n]['accuracy'],
        lda_results['accuracy'],
        chi2_results[best_chi2_k]['accuracy']
    ],
    'Precision': [
        precision_base,
        pca_results[best_pca_n]['precision'],
        lda_results['precision'],
        chi2_results[best_chi2_k]['precision']
    ],
    'Recall': [
        recall_base,
        pca_results[best_pca_n]['recall'],
        lda_results['recall'],
        chi2_results[best_chi2_k]['recall']
    ],
    'F1-Score': [
        f1_base,
        pca_results[best_pca_n]['f1_score'],
        lda_results['f1_score'],
        chi2_results[best_chi2_k]['f1_score']
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("\nComparison Summary:")
print("=" * 100)
print(comparison_df.to_string(index=False))
print("=" * 100)

# Save comparison results
comparison_df.to_csv('feature_engineering_comparison.csv', index=False)
print("\n✓ Comparison results saved to 'feature_engineering_comparison.csv'")


# STEP 8: DETAILED EVALUATION OF BEST METHOD

print("\n" + "=" * 80)
print("STEP 8: DETAILED EVALUATION OF BEST METHOD")
print("=" * 80)

# Determine best method based on accuracy
best_method_idx = comparison_df['Accuracy'].idxmax()
best_method = comparison_df.loc[best_method_idx, 'Method']
best_accuracy = comparison_df.loc[best_method_idx, 'Accuracy']

print(f"\nBest Feature Engineering Method: {best_method}")
print(f"Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")

# Get the best model and data
if 'PCA' in best_method:
    X_best = pca_results[best_pca_n]['X_transformed']
    best_model = pca_results[best_pca_n]['model']
    method_type = 'PCA'
elif 'LDA' in best_method:
    X_best = lda_results['X_transformed']
    best_model = lda_results['model']
    method_type = 'LDA'
elif 'Chi-Square' in best_method:
    X_best = chi2_results[best_chi2_k]['X_transformed']
    best_model = chi2_results[best_chi2_k]['model']
    method_type = 'Chi-Square'
else:
    X_best = X_tfidf_dense
    best_model = rf_clf_base
    method_type = 'Baseline'

# Split data
X_train_best, X_test_best, y_train_best, y_test_best = train_test_split(
    X_best, y, test_size=0.2, random_state=42, stratify=y
)

# Train final model
best_model.fit(X_train_best, y_train_best)
y_pred_best = best_model.predict(X_test_best)

# Print classification report
print("\n" + "=" * 80)
print("CLASSIFICATION REPORT (BEST METHOD)")
print("=" * 80)
print(classification_report(y_test_best, y_pred_best,
                          target_names=le_label.classes_))

# Confusion Matrix
cm = confusion_matrix(y_test_best, y_pred_best)
print("\n" + "=" * 80)
print("CONFUSION MATRIX (BEST METHOD)")
print("=" * 80)
print(cm)
print("\nRows: Actual classes")
print("Columns: Predicted classes")
print(f"Classes: {le_label.classes_}")


# STEP 9: SAVE RESULTS AND VISUALIZATIONS

print("\n" + "=" * 80)
print("STEP 9: SAVING RESULTS")
print("=" * 80)

# Save detailed results
results_summary = {
    'Best Method': best_method,
    'Best Accuracy': best_accuracy,
    'Baseline Accuracy': accuracy_base,
    'Improvement': best_accuracy - accuracy_base,
    'Original Features': X_tfidf_dense.shape[1],
    'Reduced Features': comparison_df.loc[best_method_idx, 'Features'],
    'Dimensionality Reduction': (1 - comparison_df.loc[best_method_idx, 'Features'] / X_tfidf_dense.shape[1]) * 100
}

print("\nResults Summary:")
for key, value in results_summary.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")


# FINAL SUMMARY

print("\n" + "=" * 80)
print("MILESTONE 2 FEATURE ENGINEERING COMPLETED SUCCESSFULLY!")
print("=" * 80)
print("\nSummary:")
print(f"- Total methods compared: {len(comparison_df)}")
print(f"- Best method: {best_method}")
print(f"- Best accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
print(f"- Original features: {X_tfidf_dense.shape[1]}")
print(f"- Reduced features: {int(comparison_df.loc[best_method_idx, 'Features'])}")
print(f"- Dimensionality reduction: {results_summary['Dimensionality Reduction']:.2f}%")
print(f"- Improvement over baseline: {(best_accuracy - accuracy_base)*100:+.2f}%")
print("\n" + "=" * 80)
print("Files created:")
print("  - feature_engineering_comparison.csv")
print("=" * 80)

MILESTONE 2: HATE SPEECH DETECTION WITH FEATURE ENGINEERING
Student: Aby Babu Alappat

STEP 1: LOADING AND PREPARING DATA
Dataset loaded successfully!
Total records: 60444
Columns: ['post_id', 'annotator_id', 'label', 'target', 'post_tokens']

After cleaning:
Total records: 60444

Label distribution:
label
normal        24449
hatespeech    18070
offensive     17925
Name: count, dtype: int64

Label encoding:
  hatespeech: 0
  normal: 1
  offensive: 2

STEP 2: TEXT FEATURE EXTRACTION USING TF-IDF
TF-IDF Feature Matrix Shape: (60444, 1000)
Number of features extracted: 1000
Sample feature names: ['able', 'abortion', 'absolutely', 'abuse', 'accept', 'according', 'account', 'act', 'acting', 'action']

STEP 3: BASELINE MODEL (NO FEATURE ENGINEERING)
Training set size: 48355
Testing set size: 12089

Training Random Forest with 1000 features...

Baseline Results (All 1000 TF-IDF Features):
  Accuracy:  0.6494 (64.94%)
  Precision: 0.6451
  Recall:    0.6494
  F1-Score:  0.6468

STEP 4: FEATURE